# Assistant Axis Reanalysis: Public Geometry, Axis Interpretation, and Forecasting Baselines

This notebook is an executable appendix for the Paper 1.5 work-in-progress. It starts from canonical pre-H100 artifacts in the full research repo and stops before execution-time H100 validation, prompt-battery generation, extraction-boundary diagnostics, RunPod logs, and forecast-vs-observed arrow tools.


## N00. Setup and Provenance

Question: which canonical pre-H100 artifacts are available for the walkthrough?

Data loaded: the clean-repo copy plan and canonical claim traceability table.

Method: use standard-library CSV/JSON loaders so the notebook can run in a minimal local Python kernel.

Result shown: artifact counts by type/status and unresolved review rows.

Caveat: this is a notebook skeleton, not a final paper or clean repo copy.


In [1]:
from pathlib import Path
import csv, json, math, statistics, textwrap

def find_repo_root():
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "research" / "RESEARCH_STATE.md").exists():
            return p
    raise RuntimeError("Could not find repo root from current working directory.")

REPO_ROOT = find_repo_root()
OUT_DIR = REPO_ROOT / "research" / "outputs" / "paper15_notebook_core"
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO_ROOT)

def rel(path):
    return Path(path)

def exists(path):
    return (REPO_ROOT / path).exists()

def load_csv(path):
    p = REPO_ROOT / path
    if not p.exists():
        print("MISSING:", path)
        return []
    with p.open(newline="", encoding="utf-8") as fh:
        return list(csv.DictReader(fh))

def load_json(path):
    p = REPO_ROOT / path
    if not p.exists():
        print("MISSING:", path)
        return None
    return json.loads(p.read_text(encoding="utf-8"))

def fnum(value, default=None):
    try:
        return float(value)
    except (TypeError, ValueError):
        return default

def table(rows, columns=None, limit=12):
    rows = list(rows or [])
    if not rows:
        print("(no rows)")
        return
    if columns is None:
        columns = list(rows[0].keys())
    clipped = rows[:limit]
    widths = {c: max(len(str(c)), *(len(str(r.get(c, ""))) for r in clipped)) for c in columns}
    print(" | ".join(str(c).ljust(widths[c]) for c in columns))
    print("-+-".join("-" * widths[c] for c in columns))
    for r in clipped:
        print(" | ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))
    if len(rows) > limit:
        print(f"... {len(rows) - limit} more rows")

def rows_by_count(rows, key):
    counts = {}
    for r in rows:
        counts[r.get(key, "")] = counts.get(r.get(key, ""), 0) + 1
    return [{key: k, "count": v} for k, v in sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))]

def role_dataframe_from_geometry(geometry):
    roles = geometry["roles"]
    records = []
    for name, coords, cluster in zip(roles["names"], roles["pca3d"], roles.get("clusters", [""] * len(roles["names"]))):
        records.append({"role": name, "pc1": coords[0], "pc2": coords[1], "pc3": coords[2], "cluster": cluster})
    for axis in ["pc1", "pc2", "pc3"]:
        vals = sorted(r[axis] for r in records)
        n = len(vals)
        for r in records:
            rank = sum(v <= r[axis] for v in vals)
            r[axis + "_percentile"] = 100 * (rank - 1) / max(n - 1, 1)
    return records

def top_bottom(rows, axis, n=15):
    top = sorted(rows, key=lambda r: r[axis], reverse=True)[:n]
    bottom = sorted(rows, key=lambda r: r[axis])[:n]
    return top, bottom

def safe_plot_pc_scatter(rows, x="pc1", y="pc2", labels=None, out_name="scatter.png"):
    try:
        import matplotlib.pyplot as plt
    except Exception as exc:
        print("matplotlib unavailable; skipping plot:", exc)
        return None
    labels = set(labels or [])
    xs = [r[x] for r in rows]
    ys = [r[y] for r in rows]
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(xs, ys, s=14, alpha=0.5)
    for r in rows:
        if r["role"] in labels:
            ax.annotate(r["role"], (r[x], r[y]), fontsize=8)
    ax.axhline(0, color="0.85", linewidth=1)
    ax.axvline(0, color="0.85", linewidth=1)
    ax.set_xlabel(x.upper())
    ax.set_ylabel(y.upper())
    ax.set_title(f"Qwen role geometry: {x.upper()} vs {y.upper()}")
    out = FIG_DIR / out_name
    fig.tight_layout()
    fig.savefig(out, dpi=160)
    plt.close(fig)
    print("Saved", out.relative_to(REPO_ROOT))
    return out


Repo root: /Users/alfred/Projects/Substack/mechonistic_interpretability/assistant-axis


In [2]:
copy_plan = load_csv("research/outputs/paper15_clean_repo_copy_plan/clean_repo_copy_plan.csv")
claim_trace = load_csv("research/outputs/paper15_clean_repo_copy_plan/canonical_claims_traceability_table.csv")
print("Copy-plan rows:", len(copy_plan))
print("Claim-trace rows:", len(claim_trace))
print("\nCanonical status counts:")
table(rows_by_count(copy_plan, "canonical_status"), ["canonical_status", "count"], limit=20)
print("\nArtifact type counts:")
table(rows_by_count(copy_plan, "artifact_type"), ["artifact_type", "count"], limit=20)
print("\nRows needing user review or unresolved status:")
review = [r for r in copy_plan if "review" in r.get("canonical_status", "") or "unresolved" in r.get("canonical_status", "")]
table(review, ["source_path", "canonical_status", "notes_or_uncertainties"], limit=12)


Copy-plan rows: 57
Claim-trace rows: 13

Canonical status counts:
canonical_status            | count
----------------------------+------
canonical_include           | 43   
optional_include            | 7    
unresolved_need_user_review | 4    
draft_reference_only        | 3    

Artifact type counts:
artifact_type               | count
----------------------------+------
output_dir                  | 16   
script                      | 7    
state_doc                   | 5    
method_note                 | 4    
processed_data              | 4    
draft                       | 3    
stress_test_report          | 3    
interpretation_note         | 2    
method_card                 | 2    
public_data_dir             | 2    
table                       | 2    
interpretation_notes_dir    | 1    
method_cards_dir            | 1    
negative_stress_test_report | 1    
public_data                 | 1    
reference                   | 1    
validation_note             | 1    
visualizati

## N01. Public Geometry and Artifact Reconstruction

Question: what public geometry and prompt artifacts anchor the analysis?

Data loaded: `research/visualizations/geometry_viz_data.json`, prompt artifact inventories, and role rollout reconstruction inventory.

Method: inspect schema, count roles/traits, summarize clusters, and verify public input basis.

Result shown: model/role counts, cluster counts, and prompt artifact component counts.

Caveat: public artifacts allow intended input reconstruction, but generated responses and response-level judge filters are not public.


In [3]:
geometry = load_json("research/visualizations/geometry_viz_data.json")
print("Geometry metadata:", geometry.get("metadata", {}))
roles_df = role_dataframe_from_geometry(geometry)
print("Role count:", len(roles_df))
print("Trait count:", len(geometry.get("traits", {}).get("names", [])))
print("\nCluster counts:")
table(rows_by_count(roles_df, "cluster"), ["cluster", "count"], limit=20)

role_artifacts = load_csv("research/outputs/prompt_artifact_inventory/role_prompt_artifact_index.csv")
trait_artifacts = load_csv("research/outputs/prompt_artifact_inventory/trait_prompt_artifact_index.csv")
rollout_inventory = load_csv("research/outputs/role_rollout_artifact_audit/role_prompt_reconstruction_inventory.csv")
artifact_summary = [
    {"component": "role prompt artifacts", "rows": len(role_artifacts), "source": "role_prompt_artifact_index.csv"},
    {"component": "trait prompt artifacts", "rows": len(trait_artifacts), "source": "trait_prompt_artifact_index.csv"},
    {"component": "role reconstruction inventory", "rows": len(rollout_inventory), "source": "role_prompt_reconstruction_inventory.csv"},
    {"component": "shared extraction questions", "rows": sum(1 for _ in (REPO_ROOT / "data/extraction_questions.jsonl").open(encoding="utf-8")) if exists("data/extraction_questions.jsonl") else "missing", "source": "data/extraction_questions.jsonl"},
]
table(artifact_summary, ["component", "rows", "source"], limit=10)
non_default = [r for r in rollout_inventory if r.get("is_default") == "False"]
combo_counts = sorted(set(r.get("theoretical_input_combinations") for r in non_default))
print("Non-default roles in rollout inventory:", len(non_default))
print("Theoretical combinations per non-default role:", combo_counts)
print("Example reconstruction row:")
table(non_default[:1], ["role", "positive_instruction_count", "global_extraction_question_count", "theoretical_input_combinations", "first_instruction"], limit=1)


Geometry metadata: {'model_used': 'GPT-5.5', 'source_model': 'Qwen/Qwen3-32B', 'vector_root': '/Users/alfred/Projects/Substack/mechonistic_interpretability/assistant-axis/downloads/hf_vectors/qwen-3-32b'}
Role count: 275
Trait count: 240

Cluster counts:
cluster                 | count
------------------------+------
procedural_professional | 126  
grounded_social         | 54   
mythic_spiritual        | 51   
combative_iconoclast    | 15   
editorial               | 13   
trickster_chaos         | 10   
other                   | 6    
component                     | rows | source                                  
------------------------------+------+-----------------------------------------
role prompt artifacts         | 276  | role_prompt_artifact_index.csv          
trait prompt artifacts        | 240  | trait_prompt_artifact_index.csv         
role reconstruction inventory | 276  | role_prompt_reconstruction_inventory.csv
shared extraction questions   | 240  | data/extraction_qu

## N02. Cross-Model Scope and Caveats

Question: how model-general are the relevant axes and broad topology?

Data loaded: cross-model PC correlations, best-match PCs, and cluster topology metrics.

Method: display Qwen-Llama correlations and ARI/NMI cluster alignment summaries.

Result shown: PC2 partly transfers inside a shared PC1/PC2 subspace; PC3 is weaker; clusters are partially conserved.

Caveat: cross-model hard clusters and same-index later PCs should not be treated as universal without alignment caveats.


In [4]:
pc_corr = load_csv("research/outputs/cross_model_pc2_pc3_diagnostic/cross_model_pc_correlation_matrix.csv")
best_matches = load_csv("research/outputs/cross_model_pc2_pc3_diagnostic/cross_model_pc_best_matches.csv")
cluster_metrics = load_json("research/outputs/cross_model_cluster_topology/cross_model_cluster_similarity_metrics.json")
ql_same = [r for r in pc_corr if r["model_a"] == "qwen" and r["model_b"] == "llama" and r["pc_a"] == r["pc_b"]]
print("Qwen-Llama same-index PC correlations:")
table(ql_same, ["pc_a", "matched_role_count", "pearson_r", "spearman_r"], limit=10)
print("\nBest matches for Qwen PCs against Llama:")
ql_best = [r for r in best_matches if r["model_a"] == "qwen" and r["model_b"] == "llama"]
table(ql_best, ["pc_a", "best_matching_pc_b", "pearson_r", "spearman_r", "matched_role_count"], limit=10)
print("\nCluster similarity metrics:")
metric_rows = cluster_metrics.get("cluster_similarity_metrics", []) if cluster_metrics else []
table(metric_rows, ["model_pair", "matched_role_count", "kmeans_top3_ari", "kmeans_top3_nmi", "kmeans_top5_ari", "kmeans_top5_nmi"], limit=10)


Qwen-Llama same-index PC correlations:
pc_a | matched_role_count | pearson_r           | spearman_r         
-----+--------------------+---------------------+--------------------
PC1  | 275                | 0.675842836108825   | 0.5860673379302387 
PC2  | 275                | 0.605808866959668   | 0.42971985805372037
PC3  | 275                | 0.44043920658341235 | 0.5578713828221921 

Best matches for Qwen PCs against Llama:
pc_a | best_matching_pc_b | pearson_r           | spearman_r          | matched_role_count
-----+--------------------+---------------------+---------------------+-------------------
PC1  | PC2                | -0.6759988408381201 | -0.7855866824384755 | 275               
PC2  | PC1                | 0.6916005766860976  | 0.6824084706153891  | 275               
PC3  | PC3                | 0.44043920658341235 | 0.5578713828221921  | 275               

Cluster similarity metrics:
model_pair     | matched_role_count | kmeans_top3_ari     | kmeans_top3_nmi     | kme

## N03. PC1 Interpretation

Question: what does PC1 separate in Qwen role geometry?

Data loaded: Qwen role PCA coordinates from public geometry.

Method: rank roles by PC1 and inspect endpoint examples.

Result shown: high-PC1 roles concentrate around evaluator/procedural/correctness pressure; low-PC1 roles are more open, symbolic, expressive, or possibility-rich.

Caveat: endpoint labels are evidence for the forcing-function interpretation, not the interpretation itself.


In [5]:
top_pc1, bottom_pc1 = top_bottom(roles_df, "pc1", 15)
print("Top PC1 roles:")
table(top_pc1, ["role", "cluster", "pc1", "pc2", "pc3"], limit=15)
print("\nBottom PC1 roles:")
table(bottom_pc1, ["role", "cluster", "pc1", "pc2", "pc3"], limit=15)
safe_plot_pc_scatter(roles_df, "pc1", "pc2", labels=["assistant", "auditor", "validator", "poet", "bard", "oracle", "demon"], out_name="qwen_pc1_pc2_key_roles.png")


Top PC1 roles:
role         | cluster                 | pc1                | pc2                 | pc3                
-------------+-------------------------+--------------------+---------------------+--------------------
auditor      | procedural_professional | 48.15501628203943  | -12.294598875527148 | 13.609156371558782 
examiner     | procedural_professional | 45.62248211313589  | -13.63690990152403  | 10.556259247627086 
evaluator    | procedural_professional | 45.086229152592466 | -10.275203492936383 | 7.30469963237495   
supervisor   | editorial               | 44.39804594681053  | -2.5890623851627366 | -0.5052266553764483
validator    | procedural_professional | 44.30645644159651  | -6.813154727968516  | 10.36523342895664  
statistician | procedural_professional | 43.07473713356422  | -8.21728392495824   | 15.73314631389742  
screener     | editorial               | 42.995156133991685 | -2.39649664133981   | 3.826972743721189  
lawyer       | procedural_professional | 42.99347

Saved

 research/outputs/paper15_notebook_core/figures/qwen_pc1_pc2_key_roles.png


PosixPath('/Users/alfred/Projects/Substack/mechonistic_interpretability/assistant-axis/research/outputs/paper15_notebook_core/figures/qwen_pc1_pc2_key_roles.png')

## N04. PC2 Interpretation

Question: does PC2 separate situated/formative/impressionable roles from integrated/stable/durable roles when PC1 is muted or cluster-conditioned?

Data loaded: muted-PC1 PC2 extremes and cluster-conditioned PC2 diagnostic outputs.

Method: display muted-PC1 extremes, diagnostic examples, and expected-direction pass rates.

Result shown: partial support, with important counterexamples.

Caveat: PC2 remains provisional, Qwen-local, and complicated by shapeshifter/chameleon/elder.


In [6]:
muted = load_csv("research/outputs/pc2_muted_pc1_extremes/pc2_muted_pc1_top_bottom.csv")
diag = load_csv("research/outputs/pc2_cluster_conditioned_extremes/pc2_diagnostic_roles_table.csv")
checks = load_csv("research/outputs/pc2_cluster_conditioned_extremes/pc2_expected_direction_checks.csv")
print("Muted-PC1 PC2 extremes:")
table(muted, ["extreme_group", "extreme_rank", "role", "cluster", "pc1", "pc2", "pc3"], limit=20)
focus = {"patient", "amateur", "tree", "hive", "philosopher", "shapeshifter", "chameleon", "elder"}
print("\nDiagnostic roles:")
table([r for r in diag if r.get("persona") in focus], ["persona", "cluster", "pc1", "pc2", "global_pc2_rank_desc", "cluster_pc2_rank_desc", "pc2_side_global", "pc2_side_cluster"], limit=20)
global_pass = sum(r.get("global_pass") == "True" for r in checks)
cluster_pass = sum(r.get("cluster_pass") == "True" for r in checks)
print(f"Expected-direction checks: global {global_pass}/{len(checks)}, cluster-relative {cluster_pass}/{len(checks)}")
table(checks, ["persona", "expected_pc2_side", "actual_global_side", "actual_cluster_side", "global_pass", "cluster_pass", "caveat"], limit=12)


Muted-PC1 PC2 extremes:
extreme_group | extreme_rank | role           | cluster                 | pc1       | pc2        | pc3       
--------------+--------------+----------------+-------------------------+-----------+------------+-----------
top_pc2       | 1            | amateur        | grounded_social         | -0.258633 | 40.070456  | -24.428541
top_pc2       | 2            | influencer     | combative_iconoclast    | 3.229311  | 40.002556  | -2.211369 
top_pc2       | 3            | patient        | grounded_social         | 0.411892  | 29.188211  | -27.329048
top_pc2       | 4            | gamer          | combative_iconoclast    | -1.515565 | 24.950026  | 24.217635 
top_pc2       | 5            | optimist       | grounded_social         | 0.231517  | 22.207882  | -31.270952
top_pc2       | 6            | podcaster      | grounded_social         | 0.844643  | 21.108976  | -9.123879 
top_pc2       | 7            | blogger        | grounded_social         | -0.32169  | 20.848444 

## N05. PC3 Interpretation

Question: does PC3 track perturbation/intervention versus stabilization/repair?

Data loaded: PC3 validation stats and Qwen role geometry.

Method: display validation correlations and top/bottom PC3 role rankings.

Result shown: PC3 has a meaningful perturbation/stabilization signal and is not reducible to moral valence.

Caveat: PC3 is weaker cross-model and the rubric scores are deterministic rather than independent human ratings.


In [7]:
pc3_stats = load_json("research/outputs/pc3_validation/pc3_validation_stats.json")
global_stats = pc3_stats.get("global", {}) if pc3_stats else {}
pc3_metric_rows = [
    {"metric": "global Pearson r", "value": global_stats.get("pearson", {}).get("r"), "p": global_stats.get("pearson", {}).get("p")},
    {"metric": "global Spearman r", "value": global_stats.get("spearman", {}).get("r"), "p": global_stats.get("spearman", {}).get("p")},
    {"metric": "within-cluster pairwise accuracy", "value": global_stats.get("pairwise_accuracy_within_cluster", {}).get("accuracy"), "p": ""},
]
partial = pc3_stats.get("partial_controlling_for_cluster", {}) if pc3_stats else {}
if partial:
    pearson = partial.get("pearson", {})
    pc3_metric_rows.append({"metric": "cluster-controlled Pearson r", "value": pearson.get("r"), "p": pearson.get("p", "")})
table(pc3_metric_rows, ["metric", "value", "p"], limit=10)
top_pc3, bottom_pc3 = top_bottom(roles_df, "pc3", 15)
print("\nTop PC3 roles:")
table(top_pc3, ["role", "cluster", "pc1", "pc2", "pc3"], limit=15)
print("\nBottom PC3 roles:")
table(bottom_pc3, ["role", "cluster", "pc1", "pc2", "pc3"], limit=15)
safe_plot_pc_scatter(roles_df, "pc2", "pc3", labels=["auditor", "debugger", "skeptic", "caregiver", "healer", "mediator", "demon"], out_name="qwen_pc2_pc3_key_roles.png")


metric                           | value               | p                     
---------------------------------+---------------------+-----------------------
global Pearson r                 | 0.5290975254435525  | 3.0650246318710855e-21
global Spearman r                | 0.5114167306744629  | 9.917446592874317e-20 
within-cluster pairwise accuracy | 0.7732617586912065  |                       
cluster-controlled Pearson r     | 0.49070418116813197 | 4.541003297882629e-18 

Top PC3 roles:
role        | cluster                 | pc1                 | pc2                 | pc3               
------------+-------------------------+---------------------+---------------------+-------------------
hacker      | combative_iconoclast    | -2.564236047209667  | 14.293777194706433  | 36.36362997770404 
cynic       | combative_iconoclast    | -21.40683523636408  | 40.71202851103861   | 36.281452987272154
saboteur    | combative_iconoclast    | 5.936886772982807   | -0.7753230886367263 | 34.77366

Saved research/outputs/paper15_notebook_core/figures/qwen_pc2_pc3_key_roles.png


PosixPath('/Users/alfred/Projects/Substack/mechonistic_interpretability/assistant-axis/research/outputs/paper15_notebook_core/figures/qwen_pc2_pc3_key_roles.png')

## N06. Trait/Persona Relationship

Question: how much of persona geometry is recoverable from trait-vector relationships?

Data loaded: trait-persona prediction stats and trait-space PCA interpretation stats.

Method: display vector-space verification, prediction metrics, and trait-only PCA alignment values.

Result shown: trait profiles strongly reconstruct persona PCs, but trait-only PCA does not collapse persona geometry into a single trait PCA explanation.

Caveat: same-space reconstruction supports layered geometry, not psychological ontology.


In [8]:
trait_pred = load_json("research/outputs/trait_persona_prediction/trait_predicts_persona_pcs_stats.json")
trait_space = load_json("research/outputs/trait_space_interpretation/trait_space_validation_stats.json")
if trait_pred:
    print("Vector space:", trait_pred.get("vector_space", {}))
    pred_rows = []
    for pc, models in trait_pred.get("models", {}).items():
        ridge = models.get("ridge", {})
        held = ridge.get("heldout_20_percent", {})
        cv = ridge.get("five_fold_cv", {})
        pred_rows.append({"pc": pc, "ridge_5fold_r2": cv.get("r2"), "ridge_heldout_r2": held.get("r2"), "heldout_pearson": held.get("pearson"), "heldout_spearman": held.get("spearman")})
    print("\nTrait-profile cosine -> persona PC ridge metrics:")
    table(pred_rows, ["pc", "ridge_5fold_r2", "ridge_heldout_r2", "heldout_pearson", "heldout_spearman"], limit=10)
if trait_space:
    print("\nTrait-only PCA explained variance:")
    print(trait_space.get("trait_pca_explained_variance", {}))
    print("\nPersona/trait PC direction cosines:")
    print(trait_space.get("persona_trait_pc_direction_cosines", {}))
    print("\nBest trait PC to persona loading match:")
    print(trait_space.get("best_trait_pc_to_persona_loading_match", {}))


Vector space: {'model': 'Qwen/Qwen3-32B', 'layer': 48, 'role_tensor_shape_example': [64, 5120], 'trait_tensor_shape_example': [64, 5120], 'mean_pooling': 'mean over 64 stored vectors per persona/trait, then L2 normalize', 'similarity': 'raw activation-space cosine between mean role and mean trait vectors'}

Trait-profile cosine -> persona PC ridge metrics:
pc  | ridge_5fold_r2     | ridge_heldout_r2   | heldout_pearson    | heldout_spearman  
----+--------------------+--------------------+--------------------+-------------------
PC1 | 0.9994149410874581 | 0.9993760053667119 | 0.9997340728810292 | 0.9992784992784995
PC2 | 0.9987298564784556 | 0.9981026681536073 | 0.9990860849172138 | 0.9981962481962484
PC3 | 0.9996031138309444 | 0.9994637917334468 | 0.9997465072448435 | 0.998773448773449 

Trait-only PCA explained variance:
{'pc1': 0.35283568247600433, 'pc2': 0.16806772977536064, 'pc3': 0.13374404483700736, 'pc1_pc2_pc3': 0.6546474570883722}

Persona/trait PC direction cosines:
{'PC1_si

## N07. Prediction-Improvement Sequence

Question: which predictive or explanatory improvement claims are traceable enough to show in the core walkthrough?

Data loaded: canonical claims traceability table and forecasting/model-comparison outputs.

Method: filter traceability rows for prediction/model/R2 claims and show status.

Result shown: traceable claims can be used; unverified remembered numbers remain excluded.

Caveat: this section intentionally avoids kitchen-sink exploration and does not backfill missing numbers from chat memory.


In [9]:
trace_rows = claim_trace
keywords = ("r2", "predict", "forecast", "semantic", "procedural", "big five", "svd", "model")
prediction_claims = [r for r in trace_rows if any(k in " ".join(str(v).lower() for v in r.values()) for k in keywords)]
print("Prediction-related traceability rows:")
table(prediction_claims, ["claim_or_number", "value", "source_file", "status", "notes"], limit=20)
unverified = [r for r in trace_rows if r.get("status") not in {"verified", "canonical"}]
print("\nNon-verified rows:")
table(unverified, ["claim_or_number", "value", "status", "notes"], limit=20)


Prediction-related traceability rows:
claim_or_number                              | value                                         | source_file                                                                                           | status            | notes                                                                  
---------------------------------------------+-----------------------------------------------+-------------------------------------------------------------------------------------------------------+-------------------+------------------------------------------------------------------------
Semantic baseline                            | mean R2 0.389397                              | research/q2_stability/qwen/outputs/shared_latent_feature_benchmark/shared_benchmark_summary.csv       | verified          | semantic_baseline row, canonical_activation_pca3d target.              
Procedural/Codex retained features           | mean R2 0.490090                          

## N08. Prompt-to-Geometry Forecasting Baseline

Question: can prompt text forecast intended persona/trait geometry before any H100 response-state validation?

Data loaded: prompt-to-geometry forecasting results and model comparison table.

Method: display best held-out role/trait results and the top held-out model-comparison rows.

Result shown: text-only forecasting is promising as an intended-address predictor.

Caveat: this is not evidence that predicted addresses match measured response activations on novel prompts; H100 validation is deferred.


In [10]:
forecast = load_json("research/outputs/prompt_to_geometry_forecasting/forecasting_results.json")
model_cmp = load_csv("research/outputs/prompt_to_geometry_forecasting/forecasting_model_comparison.csv")
if forecast:
    print("Forecasting dataset counts:", {k: forecast.get(k) for k in ["dataset_rows", "trait_count", "role_count", "trait_holdout_count", "role_holdout_count"]})
    print("\nBest trait heldout:")
    print(json.dumps(forecast.get("best_trait_heldout", {}), indent=2)[:1200])
    print("\nBest role heldout:")
    print(json.dumps(forecast.get("best_role_heldout", {}), indent=2)[:1200])
heldout = [r for r in model_cmp if r.get("split") == "heldout"]
def mean_r2(row): return fnum(row.get("mean_R2"), -999)
heldout_sorted = sorted(heldout, key=mean_r2, reverse=True)
print("\nTop held-out model comparison rows:")
table(heldout_sorted, ["concept_type", "variant", "model", "PC1_R2", "PC2_R2", "PC3_R2", "mean_R2"], limit=12)


Forecasting dataset counts: {'dataset_rows': 2575, 'trait_count': 240, 'role_count': 275, 'trait_holdout_count': 40, 'role_holdout_count': 55}

Best trait heldout:
{
  "concept_type": "trait",
  "variant": "leakage_control",
  "model": "elastic_net_tfidf",
  "split": "heldout",
  "PC1_R2": 0.4141886666383784,
  "PC1_Pearson_r": 0.655652186951053,
  "PC1_Spearman_r": 0.6581613508442777,
  "PC1_RMSE": 33.94695070366985,
  "PC2_R2": 0.30361602902953866,
  "PC2_Pearson_r": 0.6017424302403971,
  "PC2_Spearman_r": 0.6360225140712946,
  "PC2_RMSE": 26.747563463618366,
  "PC3_R2": 0.4499792301938743,
  "PC3_Pearson_r": 0.7081589027571247,
  "PC3_Spearman_r": 0.7003752345215761,
  "PC3_RMSE": 24.6077661532339,
  "mean_R2": 0.3892613086205971
}

Best role heldout:
{
  "concept_type": "role",
  "variant": "leakage_control",
  "model": "elastic_net_tfidf",
  "split": "heldout",
  "PC1_R2": 0.7834666301347482,
  "PC1_Pearson_r": 0.8870565884276302,
  "PC1_Spearman_r": 0.893867243867244,
  "PC1_RMSE

## N09. Main Visualization Tool

Question: where is the canonical local persona geometry viewer?

Data loaded: file-existence check for `research/visualizations/persona_geometry_explorer.html`.

Method: print path and size.

Result shown: the main viewer path for local inspection.

Caveat: H100 forecast-observed arrow viewers are excluded from this notebook and no visualization files are modified.


In [11]:
viewer = REPO_ROOT / "research/visualizations/persona_geometry_explorer.html"
print("Main viewer exists:", viewer.exists())
if viewer.exists():
    print("Relative path:", viewer.relative_to(REPO_ROOT))
    print("Size bytes:", viewer.stat().st_size)
print("Excluded from this notebook: H100 forecast-observed arrow viewers and prompt-battery visualizations.")


Main viewer exists: True
Relative path: research/visualizations/persona_geometry_explorer.html
Size bytes: 458932
Excluded from this notebook: H100 forecast-observed arrow viewers and prompt-battery visualizations.


## N10. Summary of Claims, Confidence, and Next Tests

Question: what should carry forward into the technical report, and what remains deferred?

Data loaded: this notebook's traceability summary.

Method: build a compact claims table.

Result shown: report-ready claims, confidence, caveats, and next tests.

Caveat: H100 validation, extraction-boundary tests, and within-role activation clouds remain outside this pre-H100 walkthrough.


In [12]:
summary_claims = [
    {"claim": "Public persona geometry and prompt artifacts are reconstructable enough for a reproducible pre-H100 walkthrough.", "evidence_artifact": "geometry_viz_data.json; prompt inventories; role rollout audit", "confidence": "high", "caveat": "No public generated responses or judge masks.", "next_test": "Instance-level 5x240 forecaster if target role is selected."},
    {"claim": "PC1 is best read as convergence pressure versus degrees of freedom.", "evidence_artifact": "Qwen PC1 rankings; forcing-function note", "confidence": "high", "caveat": "Interpretive, not causal proof.", "next_test": "Prompt-level judge rubric validation."},
    {"claim": "PC2 is situated/formative/impressionable versus integrated/stable, provisionally.", "evidence_artifact": "muted-PC1 and cluster-conditioned PC2 diagnostics", "confidence": "medium-low", "caveat": "Shapeshifter/chameleon/elder counterexamples; cross-model rotation.", "next_test": "Blinded within-cluster matched-pair PC2 study."},
    {"claim": "PC3 tracks perturbation/intervention versus stabilization/repair.", "evidence_artifact": "pc3_validation_stats.json", "confidence": "medium", "caveat": "Cross-model PC3 weaker; deterministic rubric.", "next_test": "Independent rater validation and response-state tests."},
    {"claim": "Trait profiles strongly reconstruct persona PCs but do not replace persona PCA.", "evidence_artifact": "trait_persona_prediction; trait_space_interpretation", "confidence": "high for reconstruction, medium for interpretation", "caveat": "Same-space geometry is not psychological ontology.", "next_test": "Layered model ablations."},
    {"claim": "Prompt-to-geometry forecasting is a pre-H100 intended-address baseline.", "evidence_artifact": "prompt_to_geometry_forecasting", "confidence": "medium", "caveat": "Not execution-time response activation validation.", "next_test": "Corrected extraction-boundary validation before broad H100 interpretation."},
]
table(summary_claims, ["claim", "confidence", "evidence_artifact", "caveat", "next_test"], limit=20)
print("\nDeferred work: D01 hook-vs-hidden-state boundary test; within-role activation cloud/variance study; judge-filter centroid comparison; instance-level 5x240 forecaster; corrected broad validation only if needed.")


claim                                                                                                            | confidence                                         | evidence_artifact                                              | caveat                                                              | next_test                                                                 
-----------------------------------------------------------------------------------------------------------------+----------------------------------------------------+----------------------------------------------------------------+---------------------------------------------------------------------+---------------------------------------------------------------------------
Public persona geometry and prompt artifacts are reconstructable enough for a reproducible pre-H100 walkthrough. | high                                               | geometry_viz_data.json; prompt inventories; role rollout audit | No public g